In [2]:
import pandas as pd
# import numpy as np

In [29]:
# Dictionary of all dataset paths
dataset_paths = {
    'BD_v4': '../data/Dengue Hematology Dataset_bd v4.csv',
    'BD_v2': '../data/dengue_dataset v2.csv',
    'Kaggle': '../data/Dengue Fever Hematological Dataset - Md Mahmudul Hasan Moon - kaggle.csv',
    'BD_v1': '../data/Dengue Hematology Dataset_bd v1.csv',
    # 'BD_v3': '../data/Dengue Hematology Dataset_bd v3.csv',
    # 'Vietnam_Pediatric': '../data/1-The value of daily platelet counts for predicting dengue shock syndrome Results from a prospective observational study of 2301 Vietnamese children with dengue.csv',
    'BD_25_rachin': '../data/Dengue_clinical_dataset - Rachin, Nure Alam Siddiki; kholilullah, Ibrahim (2025).csv',
    'BD_24':'../data/Dengue diseases dataset - Mim, Md. Minhajul Hayat ; Assaduzzaman, Md; Mahmud, Arif ; Nirob, Md Asraful Sharker (2024).csv',
}

datasets = {}
dataset_info = []

# Load each dataset and collect its shape (rows, columns)
for name, path in dataset_paths.items():
    try:
                # Conditionally load based on file extension
        if path.lower().endswith('.csv'):
            df = pd.read_csv(path)
        elif path.lower().endswith(('.xlsx', '.xls')):
            df = pd.read_excel(path)
        else:
            raise ValueError(f"Unsupported file format for: {path}")

        datasets[name] = df
        
        # 1. Safely calculate Age Ratio
        age_col = df.get('Age', df.get('age', df.get('Age (year)')))
        if age_col is not None:
            age_ratio = f"Adult {(age_col > 18).sum()} / Child {(age_col <= 18).sum()}"
        else:
            age_ratio = "No Age Column"

        # 2. Safely calculate Result / Outcome Ratio
        target_col = df.get('Dengue NS1', df.get('Result', df.get('shock', df.get('Outcome'))))
        if target_col is not None:
            # We convert to string & lowercase to safely catch 'Positive', 'Yes', '1', etc.
            target_str = target_col.astype(str).str.lower()
            pos_count = target_str.isin(['positive', 'yes', '1']).sum()
            neg_count = target_str.isin(['negative', 'no', '0']).sum()
            result_ratio = f"Pos {pos_count} / Neg {neg_count}"
        else:
            result_ratio = "No Target Column"

        dataset_info.append({
            'Dataset Name': name,
            'Rows': df.shape[0],
            'Columns': df.shape[1],
            'features': set(df.columns),
            'age_ratio': age_ratio,
            'result_ratio': result_ratio  # <--- Added here
        })
    except Exception as e:
        dataset_info.append({
            'Dataset Name': name,
            'Rows': 'Error',
            'Columns': str(e),
            'features': None,
            'age_ratio': None,
            'result_ratio': None
        })

# Display the summary 
summary_df = pd.DataFrame(dataset_info)
print(summary_df)



   Dataset Name  Rows  Columns  \
0         BD_v4  1329       13   
1         BD_v2    50       10   
2        Kaggle  1523       19   
3         BD_v1  1037       13   
4  BD_25_rachin  1018       13   
5         BD_24  1003        9   

                                            features               age_ratio  \
0  {AST, RBC, Sex, Neut %, Lymph %, PLT, Age, ALT...  Adult 1103 / Child 226   
1  {RBC, Sex, Neut %, Lymph %, PLT, HCT, Age (yea...      Adult 40 / Child 9   
2  {MCV(fl), HCT(%), PDW(%), MCHC(g/dl), Eosinoph...   Adult 1454 / Child 69   
3  {AST, RBC, Sex, Neut %, Lymph %, PLT, ALT, HCT...   Adult 870 / Child 167   
4  {Rash, Muscle_Pain, Outcome, Age, Headache, Pl...   Adult 820 / Child 198   
5  {Sex, WBC Count, Differential Count, Age, Plat...   Adult 884 / Child 119   

         result_ratio  
0   Pos 972 / Neg 357  
1      Pos 42 / Neg 8  
2  Pos 1042 / Neg 481  
3   Pos 805 / Neg 232  
4   Pos 697 / Neg 321  
5    No Target Column  
